In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from rich import print
# サブグラフを構築
class SubgraphState(TypedDict):
    raw_text: str # 未加工のテキスト
    stripped_text: str # 前後の空白を除去したテキスト
    punctuated_text: str # 文末に句点を追加したテキスト

def subgraph_strip_node(state: SubgraphState) -> SubgraphState:
    raw_text = state["raw_text"]
    stripped_text = raw_text.strip()

    return {
        "stripped_text": stripped_text
    }

def subgraph_punctuate_node(state: SubgraphState) -> SubgraphState:
    stripped_text = state["stripped_text"]
    punctuated_text = stripped_text + "。"

    return {
        "punctuated_text": punctuated_text
    }

builder = StateGraph(state_schema=SubgraphState)
builder.add_node("subgraph_strip_node", subgraph_strip_node)
builder.add_node("subgraph_punctuate_node", subgraph_punctuate_node)

builder.add_edge(START, "subgraph_strip_node")
builder.add_edge("subgraph_strip_node", "subgraph_punctuate_node")
builder.add_edge("subgraph_punctuate_node", END)

subgraph = builder.compile(checkpointer=True)

# 親グラフを構築
class ParentState(TypedDict):
    input_text: str # 入力された未加工のテキスト
    cleaned_text: str # クリーニング後のテキスト

def call_subgraph(state: ParentState) -> ParentState:
    input_text = state["input_text"]

    res = subgraph.invoke({"raw_text": input_text})
    cleaned_text = res["punctuated_text"]
    return {
        "cleaned_text": cleaned_text
    }

builder = StateGraph(state_schema=ParentState)
builder.add_node("call_subgraph", call_subgraph)

builder.add_edge(START, "call_subgraph")
builder.add_edge("call_subgraph", END)

parent_graph = builder.compile()

input_text = "   LangGraph は本当に面白い        "
for chunk in parent_graph.stream(
    {"input_text": input_text},
    # サブグラフのデータストリーミング出力を有効化 => サブグラフのデータを全体のデータに追加する
    subgraphs=True,
    stream_mode=["updates"]
):
    print(chunk)

from IPython.display import display, Image
display(
    Image(
        parent_graph
        .get_graph(xray=True)
        .draw_mermaid_png()
    )
)